In [36]:
import glob
import os
import pandas as pd
from lxml import etree
from collections import defaultdict

## Functions

In [37]:
# namespace
ns = {"tei": "http://www.tei-c.org/ns/1.0", "xmlns:rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#"}

# Processing instructions
model_pi = etree.ProcessingInstruction(
    "xml-model",
    'href="https://raw.githubusercontent.com/cceh/FormierungEuropas/refs/heads/main/CEI_TEI_Schema/tei_cei_formierung_edited.rng" '
    'type="application/xml" schematypens="http://relaxng.org/ns/structure/1.0"',
)
stylesheet_pi = etree.ProcessingInstruction(
    "xml-stylesheet",
    'type="text/css" href="https://raw.githubusercontent.com/cceh/FormierungEuropas/refs/heads/main/CEI_TEI_Schema/style.css"',
)

def parse_template(template_path):
    tree = etree.parse(template_path)
    root = tree.getroot()
    xml_template = etree.tostring(root, pretty_print=True, encoding="unicode")
    return xml_template

def safe_get(row, col):
    return row[col] if col in row.index and pd.notna(row[col]) else None

def transform(template_path, source, df, data):
    xml_template = parse_template(template_path)

    for _, row in df.iterrows():
        template = etree.fromstring(xml_template)
        
        tei_ns = "{" + f"{ns['tei']}" + "}"
        idno = f"{row['Band']}, S. {row['Seite']}, Nr. {row['Nummer']}" 

        # --- TEI[@source] ---
        template.attrib["source"] = source

        # --- Title ---
        title = template.xpath("//tei:titleStmt/tei:title", namespaces=ns)[0]
        title.text = idno

        # --- Source Description ---
        # bibl (as first bibl)
        idno_bibl = etree.Element(tei_ns+"bibl")
        idno_bibl.text = idno
        source_desc = template.xpath("//tei:sourceDesc", namespaces=ns)[0]
        source_desc.insert(0, idno_bibl)
        
        # JL-Number
        part = safe_get(row, "JL")
        if part:
            jl_nummer = str(row['JL']).split() # warum split? Wegen möglicher Zusatzangaben, die ignoriert werden sollen? Was passiert bei "JL II, S. 315"?
            jl = template.xpath("//tei:sourceDesc/tei:bibl[@type='jaffe']", namespaces=ns)[0]
            jl.text = jl_nummer[0]
        
        # --- diploDesc ---
        # authen
        part = safe_get(row, "Echtheit")
        if part:
            authen = template.xpath("//tei:authen/tei:p", namespaces=ns)[0]
            authen.text = f"{part}"
#            authen.attrib['n'] =
        
        # Issue place
        part = safe_get(row, "Ort")
        if part:
            issuePlace = template.xpath("//tei:issued/tei:placeName", namespaces=ns)[0]
            issuePlace.text = f"{part}"
        # Issue date
        part = safe_get(row, "Datum")
        if part:
            issueDate = template.xpath("//tei:issued/tei:date", namespaces=ns)[0]
            issueDate.text = f"{part}"
            # nummerical date
            part = safe_get(row, "notBefore")
            if part:
                issueDate.attrib['notBefore'] = f"{part}"
            part = safe_get(row, "notAfter")
            if part:
                issueDate.attrib['notAfter'] = f"{part}"
        
#        legal actor...
#        objectType / refPoint...

        # --- body ---
        # Abstract
        part = safe_get(row, "Kurzregest")
        if part:
            abstract = template.xpath("//tei:div[@type='abstract']/tei:p", namespaces=ns)[0]
            abstract.text = f"{part}"
        # Incipit
        part = safe_get(row, "Incipit")
        if part:
            diploPart = template.xpath("//tei:diploPart[@type='incipit']", namespaces=ns)[0]
            diploPart.text = f"{part}"
        # Subscriptio
        part = safe_get(row, "Subscriptio")
        if part:
            diploPart_item = template.xpath("//tei:diploPart[@type='subscriptio']/tei:list/tei:item", namespaces=ns)[0]
            diploPart_item.text = f"{part}"
        # Datatio
        part = safe_get(row, "Datierung")
        if part:
            diploPart = template.xpath("//tei:diploPart[@type='datatio']", namespaces=ns)[0]
            diploPart.text = f"{part}"
        
        # --- Transmission/Literature ---
        # Original
        part = safe_get(row, "Original")
        if part:
            original = template.xpath("//tei:div[@type='msSources']//tei:bibl[@type='original']", namespaces=ns)[0]
            original.text = f"{part}"
        # Copies
        part = safe_get(row, "Kopie")
        if part:
            copy = template.xpath("//tei:div[@type='msSources']//tei:bibl[@type='msCopy']", namespaces=ns)[0]
            copy.text = f"{part}"
        # Prints
        part = safe_get(row, "Druck")
        if part:
            ed_print = template.xpath("//tei:div[@type='msSources']//tei:bibl[@type='print']", namespaces=ns)[0]
            ed_print.text = f"{part}"
        
#        Decretals
#        Historiography

        # Editions
        # data source as first Edition (may be needed to put as first regest?!)
        edition1 = template.xpath("//tei:div[@type='editions']//tei:bibl", namespaces=ns)[0]
        edition1.text = idno
        part = safe_get(row, "Edition")
        if part:
            editions2 = template.xpath("//tei:div[@type='editions']//tei:bibl", namespaces=ns)[1]
            editions2.text = f"{part}"
        # JL as first Regest
        part = safe_get(row, "JL")
        if part:
            regestaJL = template.xpath("//tei:div[@type='regesta']//tei:bibl", namespaces=ns)[0]
            regestaJL.text = f"JL {part}"
        # Regesta
        part = safe_get(row, "Regest")
        if part:
            regesta2 = template.xpath("//tei:div[@type='regesta']//tei:bibl", namespaces=ns)[1]
            regesta2.text = f"{part}"
            
        # --- Comentary/Bibliography ---
#        Diplomatic commentary (diploDesc?)
        
        # Commentary
        part = safe_get(row, "Sachkommentar")
        if part:
            commentary = template.xpath("//tei:div[@type='commentary']/tei:p", namespaces=ns)[0]
            commentary.text = f"{part}"
        # Zit
        part = safe_get(row, "Zit")
        if part:
            zitlit1 = template.xpath("//tei:div[@type='bibliography']/tei:listBibl/tei:bibl", namespaces=ns)[0]
            zitlit1.text = f"{part}"
        # Vgl
        part = safe_get(row, "Vgl")
        if part:
            zitlit2 = template.xpath("//tei:div[@type='bibliography']/tei:listBibl/tei:bibl", namespaces=ns)[1]
            zitlit2.text = f"{part}"

### Beginn Schönfeld-Daten: Rota Inschrift, Rota-Devise, Benevalete, Komma
        # as Diplomatischer Kommentar
        diploDesc_p = template.xpath("//tei:body/tei:div/tei:diploDesc/tei:p", namespaces=ns)[0]
        item_list = etree.Element(tei_ns+"list")
        part = safe_get(row, "Rota-Inschrift")
        if part:
                item = etree.Element(tei_ns+"item")
                item.text = f"Rota-Inschrift: {part}"
                item_list.insert(0, item)
        part = safe_get(row, "Rota-Devise")
        if part:
            item = etree.Element(tei_ns+"item")
            item.text = f"Rota-Devise: {part}"
            item_list.insert(1, item)
        part = safe_get(row, "BV")
        if part:
            item = etree.Element(tei_ns+"item")
            item.text = f"Bene Valete: {part}"
            item_list.insert(2, item)
        part = safe_get(row, "K")
        if part:
            item = etree.Element(tei_ns+"item")
            item.text = f"Komma: {part}"
            item_list.insert(3, item)
        if len(item_list):
            diploDesc_p.insert(0, item_list)
### Ende Schönfeld-Daten

        # Editionstext
        part = safe_get(row, "Edition Text")
        if part:
            note = template.xpath("//tei:div[@type='note']/tei:p", namespaces=ns)[0]
            edition = etree.Element(tei_ns+"p")
            edition.text = f"{part}"
            note.addnext(edition)
        
        
        data.append(template)
        template = None # Reset template

    print(len(data))
    return data

def save_xml_files(save_path, data):
    file_names = set()
    counter = defaultdict(int)
    for regest in data:
        datenquelle = regest.xpath("//tei:title", namespaces=ns)[0]
        datenquelltext = datenquelle.text.strip().replace(".", "").replace(",", "").replace(" ", "_")
        # make sure each file name exists only once (!only per .tsv file!)
        if datenquelltext in file_names:
            counter[datenquelltext] += 1
            datenquelltext = f"{datenquelltext}-{counter[datenquelltext]}"
        else:
            file_names.add(datenquelltext)
        full_path = f'{save_path}{datenquelltext}.xml'
        tree = etree.ElementTree(regest)
        # add processing instructions
        tree.getroot().addprevious(model_pi)
        tree.getroot().addprevious(stylesheet_pi)
        tree.write(full_path, encoding="UTF-8", xml_declaration=True)
        print(full_path)
    print(counter)
    return counter

# Main
def populate_templates(template_path, source, data_directory, save_path, file_list=None, data_type='csv'):
    if file_list:
        files = [os.path.join(data_directory + f) for f in file_list]
        print(files)
    else:
        files = glob.glob(os.path.join(data_directory + f"*.{data_type}"))
    n = 0
    idno_variants = []
    for filename in files:
        data = []
        print(filename.split("/")[-1])
        if data_type == 'csv':
            df = pd.read_csv(filename, sep='\t')
        elif data_type == 'xlsx':
            df = pd.read_excel(filename)
        data = transform(template_path, source, df, data)
        idnos = save_xml_files(save_path, data)
        idno_variants.append(idnos)
        n += len(data)
    print(f"{n} Regesta transformed")
    return data, idno_variants


## Inspect Regesta-table
Lfd. Nummer | Band | Seite | Nummer | Echtheit | Ort | Datum | notBefore | notAfter | Kurzregest | Stop1 | Original | Kopie | Druck | Edition | Regest | JL | Vgl | Zit | Sachkommentar | Stop2 | ...


Mit Tabs (Schönfeld):  
Lfd. Nummer	Band	Seite	Nummer	Echtheit	Ort	Datum	notBefore	notAfter	Kurzregest	Stop1	Original	Kopie	Druck	Edition	Regest	JL	Vgl	Zit	Sachkommentar	Stop2	Incipit	Stop3	Edition Text	Stop4	Rota-Inschrift	Rota-Devise	BV	K	Subskr.	Datierung	text_part_23	text_part_24

In [ ]:
files = [
#    "Anglia_Pontificia.tsv",
#    "Germania_Pontificia.tsv",
#    "Italia_Pontificia.tsv",
#    "Oriens_Pontificus.tsv",

#    "PUU_England.tsv",
#    "PUU_Frankreich_AF.tsv",
#    "PUU_Frankreich_NF.tsv",
#    "PUU_Niederlande.tsv",
#    "PUU_Spanien.tsv",

#    "Dolle_Papsturkunden.tsv",
#    "Schoenfeld_Gegenpaepste.tsv",
#    "Holtzmann_Kanonistische_Ergaenzungen.tsv"
]

## Populate Pontificia

In [40]:
template_path = "../FormierungEuropas/01_Template_Regest.xml"

###
### 1. unter Source den richtigen Ordner festlegen - für die Datenbank?
### 2. prüfen, ob alle Spalten richtig erfasst werden!
###
source = "formierung-europas_pontificia"
data_directory = "../FormierungEuropas/Z-TSV/"
file_list = [
    "Anglia_Pontificia.tsv",
    "Germania_Pontificia.tsv",
    "Italia_Pontificia.tsv",
    "Oriens_Pontificus.tsv",
]
save_path = "../Sondierung/Pontificien/"
data, idno_variants = populate_templates(template_path, source, data_directory, save_path, file_list, data_type='csv')

['../FormierungEuropas/Z-TSV/Anglia_Pontificia.tsv', '../FormierungEuropas/Z-TSV/Germania_Pontificia.tsv', '../FormierungEuropas/Z-TSV/Italia_Pontificia.tsv', '../FormierungEuropas/Z-TSV/Oriens_Pontificus.tsv']
Anglia_Pontificia.tsv
192
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_132_Nr_88.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_132_Nr_89.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_132-133_Nr_90.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_133_Nr_91.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_133_Nr_92.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_133_Nr_93.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_133_Nr_94.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_133-134_Nr_95.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_134_Nr_96.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_134_Nr_97.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_134-135_Nr_98.xml
../Sondierung/Pontificien/Anglia_Pont_Sub_1_S_135_Nr_99.xml
../Sondierung/Pontificien/Angli

## Populate PUU

In [41]:
template_path = "../FormierungEuropas/01_Template_Regest.xml"

###
### 1. unter Source den richtigen Ordner festlegen - für die Datenbank?
### 2. prüfen, ob alle Spalten richtig erfasst werden!
###
source = "formierung-europas_PUU"
data_directory = "../FormierungEuropas/Z-TSV/"
file_list = [
    "PUU_England.tsv",
    "PUU_Frankreich_AF.tsv",
    "PUU_Frankreich_NF.tsv",
    "PUU_Niederlande.tsv",
    "PUU_Spanien.tsv",
]
save_path = "../Sondierung/PUU/"
data, idno_variants = populate_templates(template_path, source, data_directory, save_path, file_list, data_type='csv')

['../FormierungEuropas/Z-TSV/PUU_England.tsv', '../FormierungEuropas/Z-TSV/PUU_Frankreich_AF.tsv', '../FormierungEuropas/Z-TSV/PUU_Frankreich_NF.tsv', '../FormierungEuropas/Z-TSV/PUU_Niederlande.tsv', '../FormierungEuropas/Z-TSV/PUU_Spanien.tsv']
PUU_England.tsv


561
../Sondierung/PUU/PUU_England_1_S_291_Nr_50.xml
../Sondierung/PUU/PUU_England_1_S_291-293_Nr_51.xml
../Sondierung/PUU/PUU_England_1_S_293-295_Nr_52.xml
../Sondierung/PUU/PUU_England_1_S_295-297_Nr_53.xml
../Sondierung/PUU/PUU_England_1_S_297-298_Nr_54.xml
../Sondierung/PUU/PUU_England_1_S_298-300_Nr_55.xml
../Sondierung/PUU/PUU_England_1_S_300-302_Nr_56.xml
../Sondierung/PUU/PUU_England_1_S_302-303_Nr_57.xml
../Sondierung/PUU/PUU_England_1_S_303-304_Nr_58.xml
../Sondierung/PUU/PUU_England_1_S_304-305_Nr_59.xml
../Sondierung/PUU/PUU_England_1_S_305-306_Nr_60.xml
../Sondierung/PUU/PUU_England_1_S_306-307_Nr_61.xml
../Sondierung/PUU/PUU_England_1_S_307-309_Nr_62.xml
../Sondierung/PUU/PUU_England_1_S_309-312_Nr_63.xml
../Sondierung/PUU/PUU_England_1_S_312-315_Nr_64.xml
../Sondierung/PUU/PUU_England_1_S_315_Nr_65.xml
../Sondierung/PUU/PUU_England_1_S_315-317_Nr_66.xml
../Sondierung/PUU/PUU_England_1_S_317-319_Nr_67.xml
../Sondierung/PUU/PUU_England_1_S_320_Nr_68.xml
../Sondierung/PUU/PU

## Populate Dolle

In [42]:
template_path = "../FormierungEuropas/01_Template_Regest.xml"

###
### 1. unter Source den richtigen Ordner festlegen - für die Datenbank?
### 2. prüfen, ob alle Spalten richtig erfasst werden!
###
source = "dolle"
data_directory = "../FormierungEuropas/Z-TSV/"
file_list = [
    "Dolle_Papsturkunden.tsv",
]
save_path = "../Sondierung/Einzelbände/"
data, idno_variants = populate_templates(template_path, source, data_directory, save_path, file_list, data_type='csv')

['../FormierungEuropas/Z-TSV/Dolle_Papsturkunden.tsv']
Dolle_Papsturkunden.tsv
45
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_180–181_Nr_72.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_182–184_Nr_73.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_184–185_Nr_74.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_185–187_Nr_75.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_187–188_Nr_76.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_188–189_Nr_77.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_190–191_Nr_78.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_191–192_Nr_79.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_192–193_Nr_80.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_193–194_Nr_81.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_194_Nr_82.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_194–195_Nr_83.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_195_Nr_84.xml
../Sondierung/Einzelbände/Dolle_Papsturkunden_S_196_Nr_85.xml
../Son

## Populate Holtzmann

In [43]:
template_path = "../FormierungEuropas/01_Template_Regest.xml"

###
### 1. unter Source den richtigen Ordner festlegen - für die Datenbank?
### 2. prüfen, ob alle Spalten richtig erfasst werden!
###
source = "holtzmann"
data_directory = "../FormierungEuropas/Z-TSV/"
file_list = [
    "Holtzmann_Kanonistische_Ergaenzungen.tsv",
]
save_path = "../Sondierung/Einzelbände/"
data, idno_variants = populate_templates(template_path, source, data_directory, save_path, file_list, data_type='csv')

['../FormierungEuropas/Z-TSV/Holtzmann_Kanonistische_Ergaenzungen.tsv']
Holtzmann_Kanonistische_Ergaenzungen.tsv
170
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_67_Nr_55.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_67-68_Nr_56.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_68-69_Nr_57.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_69_Nr_58.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_69-70_Nr_59.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_70-71_Nr_60.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_71-72_Nr_61.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_72-73_Nr_62.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_73_Nr_63.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_74_Nr_64.xml
../Sondierung/Einzelbände/Holtzmann_Kanonistische_Ergänzungen_S_74-75_Nr_65.xml
../Sondierung/E